In [1]:
!rm -f /kaggle/working/wheels

In [2]:
import subprocess, sys, shutil
from pathlib import Path
from collections import defaultdict

WHEELS_DIR = Path("/kaggle/working/wheels")
shutil.rmtree(WHEELS_DIR, ignore_errors=True)
WHEELS_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# STEP 1: INSTALL CLEAN ENV (ONLINE)
# =========================
BASE_PACKAGES = [
    "vllm==0.17.1",
    "transformers==4.56.0",
    "rapidfuzz>=3.0.0",
    "protobuf<6",
    "peft>=0.15.0",
    "accelerate>=1.0.0",
    "bitsandbytes>=0.45.0",
]

print("▶ Installing resolved environment...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--upgrade",
    "--no-cache-dir",
    *BASE_PACKAGES,
])

# =========================
# STEP 2: FREEZE EXACT VERSIONS
# =========================
print("\n▶ Freezing environment...")

freeze = subprocess.check_output([
    sys.executable, "-m", "pip", "freeze"
]).decode().splitlines()

# ── Save FULL freeze (for debugging / audit) ──
(WHEELS_DIR / "requirements_full.txt").write_text("\n".join(freeze))

# ── Save LOCKED freeze (target packages only) ──
TARGET_PKGS = [
    "vllm", "transformers", "rapidfuzz", "protobuf",
    "huggingface-hub", "msgspec",
    "peft", "accelerate", "bitsandbytes"
]

def keep(line):
    return any(line.lower().startswith(pkg) for pkg in TARGET_PKGS)

locked = [line for line in freeze if keep(line)]
REQ_FILE = WHEELS_DIR / "requirements_locked.txt"
REQ_FILE.write_text("\n".join(locked))

print("\nLocked packages:")
for l in locked:
    print(f"  {l}")

# =========================
# STEP 3: DOWNLOAD EXACT WHEELS
# =========================
print("\n▶ Downloading exact wheels...")

subprocess.check_call([
    sys.executable, "-m", "pip", "download",
    "--dest", str(WHEELS_DIR),
    "--no-cache-dir",
    "-r", str(REQ_FILE),
])

# =========================
# STEP 4: RESOLVE DUPLICATES  ← was only "warning" before
# =========================
wheels = sorted(WHEELS_DIR.glob("*.whl"))
print(f"\n✓ Downloaded {len(wheels)} wheels")

pkg_map = defaultdict(list)
for w in wheels:
    pkg = w.name.split("-")[0].lower()
    pkg_map[pkg].append(w)

# Build a lookup from locked requirements: pkg → exact version string
locked_lookup = {}
for line in locked:
    if "==" in line:
        name, ver = line.lower().split("==", 1)
        locked_lookup[name] = ver

for pkg, files in pkg_map.items():
    if len(files) > 1:
        print(f"\n[DUPLICATE] {pkg}:")
        for f in files:
            print(f"   {f.name}")

        # Prefer the version that matches the locked freeze
        to_keep = None
        if pkg in locked_lookup:
            for f in files:
                if locked_lookup[pkg] in f.name:
                    to_keep = f
                    break

        # Fallback: keep the last (usually highest version)
        if to_keep is None:
            to_keep = files[-1]

        for f in files:
            if f is not to_keep:
                print(f"   ✗ Removing: {f.name}")
                f.unlink()
            else:
                print(f"   ✓ Keeping:  {f.name}")

# Also remove .tar.gz duplicates for non-wheel source dists
tars = sorted(WHEELS_DIR.glob("*.tar.gz"))
pkg_map_tar = defaultdict(list)
for t in tars:
    # tar names: package-version.tar.gz
    pkg = t.name.rsplit("-", 2)[0].lower()
    pkg_map_tar[pkg].append(t)

for pkg, files in pkg_map_tar.items():
    # If we already have a wheel for this package, remove the tar
    if pkg in pkg_map:
        for f in files:
            print(f"   ✗ Removing redundant sdist: {f.name}")
            f.unlink()

final_count = len(list(WHEELS_DIR.iterdir()))
print(f"\n✓ Wheelhouse ready — {final_count} files (clean dependency graph)")

▶ Installing resolved environment...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 101.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 2.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of grpcio-reflection to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-reflection to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 220.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 144.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 244.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 174.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 253.1 MB/s eta 0:00:00
   ━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-adk 1.25.1 requires opentelemetry-api<1.40.0,>=1.36.0, but you have opentelemetry-api 1.41.1 which is incompatible.
google-adk 1.25.1 requires opentelemetry-sdk<1.40.0,>=1.36.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.41.1 which is incompatible.



▶ Freezing environment...

Locked packages:
  accelerate==1.13.0
  bitsandbytes==0.49.2
  msgspec==0.21.1
  peft==0.19.1
  protobuf==5.29.6
  RapidFuzz==3.14.5
  transformers==4.56.0
  vllm==0.17.1

▶ Downloading exact wheels...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 127.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 215.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 159.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 198.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 167.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 229.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 210.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 156.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 197.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions

In [3]:
import os

print("▶ Disk usage summary:")
result = subprocess.run(["du", "-sh", str(WHEELS_DIR)],
                        capture_output=True, text=True)
print(result.stdout)

print("\n▶ Wheel directory contents:")
for w in sorted(WHEELS_DIR.glob("*.whl")):
    print(f"  {w.name}")

print("\n✓ Setup complete. Commit this notebook to save outputs as Kaggle dataset.")

▶ Disk usage summary:
4.9G	/kaggle/working/wheels


▶ Wheel directory contents:
  accelerate-1.13.0-py3-none-any.whl
  aiohappyeyeballs-2.6.1-py3-none-any.whl
  aiohttp-3.13.5-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
  aiosignal-1.4.0-py3-none-any.whl
  annotated_doc-0.0.4-py3-none-any.whl
  annotated_types-0.7.0-py3-none-any.whl
  anthropic-0.97.0-py3-none-any.whl
  anyio-4.13.0-py3-none-any.whl
  apache_tvm_ffi-0.1.10-cp312-abi3-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
  astor-0.8.1-py2.py3-none-any.whl
  attrs-26.1.0-py3-none-any.whl
  bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl
  blake3-1.0.8-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
  cachetools-7.1.0-py3-none-any.whl
  cbor2-6.0.1-cp312-cp312-manylinux_2_28_x86_64.whl
  certifi-2026.4.22-py3-none-any.whl
  cffi-2.0.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl
  charset_normalizer-3.4.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.